
LLM ZOOM CAMP - VECTOR SEARCH CODING RESULTS


In [21]:
from embedder import Embedder

embed = Embedder()

q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1 = embed.encode(q1)
v2 = embed.encode(q2)
dv = embed.encode(d)

In [22]:
v1.dot(dv)

np.float64(0.3233238326017993)

In [23]:
v2.dot(dv)

np.float64(0.019730417971579928)

Q1. Embedding a query

Embed the following query:

How does approximate nearest neighbor search work?

The embedder returns a vector of 384 numbers. What's the first value (v[0])?



In [35]:
from embedder import Embedder
embedder = Embedder()
query = "How does approximate nearest neighbor search work?"

In [117]:
v = embedder.encode(query)

Answer:

In [121]:
print(v[0])

-0.044614335778659704


In [ ]:
Q2. Cosine similarity

Question:

Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content,
and compute the cosine similarity with the query vector from Q1.


In [6]:
%pip install gitsource

/Users/haleemathsameena/Documents/Zoomcamp/llm-zoomcamp-onnx/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [12]:
from gitsource import GithubRepositoryDataReader

In [115]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [125]:
filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"

doc = next(d for d in documents if d["filename"] == filename)


page_embedding = embedder.encode(doc["content"])
similarity = v.dot(page_embedding)
print(similarity)

-0.08554717757866476


In [31]:
v_query.dot(dv)

np.float64(0.361070280302606)

Q3. Chunking and search by hand

Instead of embedding an entire lesson page, we'll split each page into smaller chunks. This usually gives better search results because each chunk focuses on a single topic.

In [57]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

contents = [chunk["content"] for chunk in chunks]

X = embedder.encode_batch(contents)


In [58]:
print(X.shape)

(295, 384)


In [59]:
scores = X.dot(v)

In [60]:
import numpy as np

best_idx = np.argmax(scores)
print(best_idx)

94


In [61]:
print(chunks[best_idx]["filename"])

02-vector-search/lessons/07-sqlitesearch-vector.md


In [56]:
print(len(chunks))
print(X.shape)
print(scores.shape)
print(best_idx)

295
(1368, 384)
(1368,)
597


In [50]:
print(scores.shape)

(1368,)


Q4. Vector search with minsearch

This time, instead of manually computing similarities with X.dot(v), we'll use VectorSearch from minsearch.

In [69]:
from minsearch import VectorSearch

vindex = VectorSearch()


In [82]:
vindex.fit(X, chunks)

In [83]:
query = "What metric do we use to evaluate a search engine?"

In [84]:
query_vector=embedder.encode(query)

In [85]:
results = vindex.search(
    query_vector=query_vector,
    num_results=5
)

In [87]:
print(results[0]['filename'])

04-evaluation/lessons/05-search-metrics.md


In [ ]:
Q5. Text search vs. vector search

In this question, you'll compare:

Vector search (VectorSearch)
Keyword search (Index)

and find which file appears in the vector search results but not in the text search results.

In [101]:
from minsearch import Index
tindex = Index(
    text_fields=["content"],
    keyword_fields=["filename", "start"]
    
)
tindex.fit(chunks)

In [90]:
query = "How do I store vectors in PostgreSQL?"

text_results = tindex.search(
    query=query,
    num_results=5
)

In [91]:
for doc in text_results:
    print(doc["filename"])

02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


In [92]:
query_vector = embedder.encode(query)

vector_results = vindex.search(
    query_vector=query_vector,
    num_results=5
)

In [93]:
for doc in vector_results:
    print(doc["filename"])

02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


In [ ]:


Q6. Hybrid Search (Reciprocal Rank Fusion)

The idea is to:

Run vector search
Run text search
Combine both ranked lists using RRF
Find the filename of the top-ranked result

In [104]:
query1 = "How do I give the model access to tools?"

text_results = tindex.search(
    query=query1,
    num_results=5
)

for doc in text_results:
    print(doc["filename"])

01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/13-function-calling.md
04-evaluation/lessons/02-ground-truth.md


In [106]:
query_vector = embedder.encode(query1)

vector_results = vindex.search(
    query_vector=query_vector,
    num_results=5
)
for doc in vector_results:
    print(doc["filename"])

01-agentic-rag/lessons/01-intro.md
04-evaluation/lessons/02-ground-truth.md
01-agentic-rag/lessons/16-other-frameworks.md
01-agentic-rag/lessons/15-frameworks.md
01-agentic-rag/lessons/13-function-calling.md


In [108]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [111]:
results = rrf([vector_results, text_results])

In [112]:
print(results[0]["filename"])

01-agentic-rag/lessons/13-function-calling.md
